# Introduction

# Import Data

In [0]:
# import ast
# import numpy as np
# import pandas as pd

# # Load core datasets as pandas DataFrames
# google_analytics = (
#     spark.read.csv(
#         "/Volumes/workspace/default/capstone_data/"
#         "clean_google_analytics_abandonment.csv",
#         header=True,
#         inferSchema=True,
#     ).toPandas()
# )

# sales = (
#     spark.read.csv(
#         "/Volumes/workspace/default/capstone_data/clean_sales.csv",
#         header=True,
#         inferSchema=True,
#     ).toPandas()
# )

# materials = (
#     spark.read.csv(
#         "/Volumes/workspace/default/capstone_data/clean_material.csv",
#         header=True,
#         inferSchema=True,
#     ).toPandas()
# )

# # Preview loaded data
# display(google_analytics.head(3))
# display(sales.head(3))
# display(materials.head(3))

# # Filter analytics events for cart and purchase logic
# event_names_of_interest = {"add_to_cart", "purchase"}
# events = google_analytics[
#     google_analytics["event_name"].isin(event_names_of_interest)
# ]

# events = events[
#     (events["event_name"] != "add_to_cart")
#     | (
#         (events["event_name"] == "add_to_cart")
#         & (events["abandoned"])
#     )
# ].sort_values("event_ts_utc", ascending=True)

# purchase_mask = events["event_name"] == "purchase"
# events.loc[purchase_mask, "abandoned"] = False


# # Assign session identifiers for add_to_cart groups and purchases
# def assign_session_id(df: pd.DataFrame) -> pd.DataFrame:
#     """Assign session_id by grouping add_to_cart and isolating purchases."""
#     local = df.copy()

#     add_mask = local["event_name"] == "add_to_cart"
#     buy_mask = local["event_name"] == "purchase"

#     add = local[add_mask].copy()
#     add["session_id"] = (
#         add.groupby(
#             [
#                 "customer_id",
#                 "device_category",
#                 "device_operating_system",
#                 "event_date",
#             ]
#         ).ngroup()
#     )

#     buy = local[buy_mask].copy()
#     base = int(add["session_id"].max()) + 1 if not add.empty else 0
#     buy = buy.reset_index(drop=True)
#     buy["session_id"] = buy.index + base

#     out = (
#         pd.concat([add, buy], axis=0)
#         .sort_values("event_ts_utc")
#         .reset_index(drop=True)
#     )
#     out["session_id"] = out["session_id"].astype(int)
#     return out


# # Build event dataset with sessions and key columns
# events = assign_session_id(events).sort_values(
#     ["customer_id", "event_ts_utc"], ascending=[False, False]
# )
# events = events[
#     ["session_id", "customer_id", "items", "event_ts_utc", "abandoned"]
# ]

# # display(events)


# # Normalize nested item structures into a flat table
# def parse_items(value: object) -> list | None:
#     """Parse items field into a list of dicts or return None."""
#     if pd.isna(value):
#         return None
#     if isinstance(value, list):
#         return value
#     if isinstance(value, dict):
#         return [value]
#     if isinstance(value, str) and value.strip():
#         try:
#             parsed = ast.literal_eval(value)
#             if isinstance(parsed, dict):
#                 return [parsed]
#             if isinstance(parsed, list):
#                 return parsed
#         except Exception:
#             return None
#     return None


# # Expand items and join material attributes
# items = events.copy()
# items["items"] = items["items"].map(parse_items)
# items = items[items["items"].notna()]
# items = items.explode("items", ignore_index=True)
# items = items[items["items"].notna()]

# items_norm = pd.json_normalize(items["items"])

# base_cols = [
#     c for c in items.drop(columns=["items"]).columns if c != "abandoned"
# ]
# ordered_cols = base_cols + list(items_norm.columns) + ["abandoned"]

# events_flat = pd.concat(
#     [items.drop(columns=["items"]), items_norm],
#     axis=1,
# )[ordered_cols]

# events_flat["item_id"] = events_flat["item_id"].astype(str)
# materials["material_id"] = materials["material_id"].astype(str)

# events_flat = (
#     events_flat.merge(
#         materials,
#         how="left",
#         left_on="item_id",
#         right_on="material_id",
#     )
#     .drop(columns=["material_id"])
# )

# # display(events_flat)

# # Engineer reference prices at multiple temporal grains
# sales["unit_profit"] = sales["profit"] / sales["physical_volume"]
# sales["unit_cost"] = sales["unit_price"] - sales["unit_profit"]

# sales["material_id"] = sales["material_id"].astype(str)
# sales["posting_date"] = pd.to_datetime(sales["posting_date"])

# sales["avg_unit_price"] = sales.groupby(
#     ["material_id", "posting_date"]
# )["unit_price"].transform("mean")
# sales["avg_unit_profit"] = sales.groupby(
#     ["material_id", "posting_date"]
# )["unit_profit"].transform("mean")
# sales["avg_unit_cost"] = sales.groupby(
#     ["material_id", "posting_date"]
# )["unit_cost"].transform("mean")

# sales["year_week"] = sales["posting_date"].dt.strftime("%Y-%U")
# sales["week_avg_unit_price"] = sales.groupby(
#     ["material_id", "year_week"]
# )["unit_price"].transform("mean")
# sales["week_avg_unit_profit"] = sales.groupby(
#     ["material_id", "year_week"]
# )["unit_profit"].transform("mean")
# sales["week_avg_unit_cost"] = sales.groupby(
#     ["material_id", "year_week"]
# )["unit_cost"].transform("mean")

# sales["year_month"] = sales["posting_date"].dt.strftime("%Y-%m")
# sales["month_avg_unit_price"] = sales.groupby(
#     ["material_id", "year_month"]
# )["unit_price"].transform("mean")
# sales["month_avg_unit_profit"] = sales.groupby(
#     ["material_id", "year_month"]
# )["unit_profit"].transform("mean")
# sales["month_avg_unit_cost"] = sales.groupby(
#     ["material_id", "year_month"]
# )["unit_cost"].transform("mean")

# sales["year"] = sales["posting_date"].dt.strftime("%Y")
# sales["year_avg_unit_price"] = sales.groupby(
#     ["material_id", "year"]
# )["unit_price"].transform("mean")
# sales["year_avg_unit_profit"] = sales.groupby(
#     ["material_id", "year"]
# )["unit_profit"].transform("mean")
# sales["year_avg_unit_cost"] = sales.groupby(
#     ["material_id", "year"]
# )["unit_cost"].transform("mean")

# # Create a de-duplicated price view with effective price backfill, including posting_date
# price_view = sales[
#     [
#         "material_id",
#         "posting_date",
#         "unit_price",
#         "avg_unit_price",
#         "week_avg_unit_price",
#         "month_avg_unit_price",
#         "year_avg_unit_price",
#         "unit_profit",
#         "avg_unit_profit",
#         "week_avg_unit_profit",
#         "month_avg_unit_profit",
#         "year_avg_unit_profit",
#         "unit_cost",
#         "avg_unit_cost",
#         "week_avg_unit_cost",
#         "month_avg_unit_cost",
#         "year_avg_unit_cost",
#     ]
# ].drop_duplicates()

# price_view["effective_unit_price"] = (
#     price_view[
#         [
#             "avg_unit_price",
#             "week_avg_unit_price",
#             "month_avg_unit_price",
#             "year_avg_unit_price",
#         ]
#     ]
#     .bfill(axis=1)
#     .iloc[:, 0]
#     .round(2)
# )

# price_view["effective_unit_profit"] = (
#     price_view[
#         [
#             "avg_unit_profit",
#             "week_avg_unit_profit",
#             "month_avg_unit_profit",
#             "year_avg_unit_profit",
#         ]
#     ]
#     .bfill(axis=1)
#     .iloc[:, 0]
#     .round(2)
# )

# price_view["effective_unit_cost"] = (
#     price_view[
#         [
#             "avg_unit_cost",
#             "week_avg_unit_cost",
#             "month_avg_unit_cost",
#             "year_avg_unit_cost",
#         ]
#     ]
#     .bfill(axis=1)
#     .iloc[:, 0]
# )

# # Drop engineered avg columns for price, profit, and cost
# cols_to_drop = [
#     "avg_unit_price", "week_avg_unit_price", "month_avg_unit_price", "year_avg_unit_price",
#     "avg_unit_profit", "week_avg_unit_profit", "month_avg_unit_profit", "year_avg_unit_profit",
#     "avg_unit_cost", "week_avg_unit_cost", "month_avg_unit_cost", "year_avg_unit_cost"
# ]
# price_view = price_view.drop(columns=cols_to_drop)

# # Group by material_id and posting_date, set effective columns to mean of each
# price_view = (
#     price_view
#     .groupby(["material_id", "posting_date"], as_index=False)[
#         ["effective_unit_price", "effective_unit_profit", "effective_unit_cost"]
#     ]
#     .mean()
#     .round(2)
# )

# # display(price_view)

# # Join effective_unit_price to events_flat by item_id=material_id and event_ts_utc (date)=posting_date
# events_flat["event_date"] = pd.to_datetime(events_flat["event_ts_utc"]).dt.date
# price_view["posting_date"] = pd.to_datetime(price_view["posting_date"]).dt.date

# full_events = events_flat.merge(
#     price_view[[
#         "material_id", "posting_date", "effective_unit_price",
#         "effective_unit_profit", "effective_unit_cost"
#     ]],
#     left_on=["item_id", "event_date"],
#     right_on=["material_id", "posting_date"],
#     how="inner"
# ).drop(columns=["material_id", "posting_date"])

# # display(full_events.sort_values(["customer_id", "event_ts_utc"], ascending=[True, False]))

# # Clean quantity to an integer vector
# qty = (
#     pd.to_numeric(full_events["quantity"], errors="coerce")
#     .fillna(1)
#     .astype("int64")
# )

# # Build repeated row positions with NumPy (compact and fast)
# pos = np.repeat(np.arange(len(full_events), dtype=np.int64), qty.to_numpy())

# # Take rows by position to avoid index alignment overhead
# item_activity_data = (
#     full_events
#     .take(pos)
#     .reset_index(drop=True)
#     .drop(columns=["quantity"])
# )
# item_activity_data["item_activity_id"] = item_activity_data.index.astype("int64")

# item_activity_data = item_activity_data[
#     [
#         "item_activity_id",
#         "session_id",
#         "customer_id",
#         "event_ts_utc",
#         "item_id",
#         "trademark",
#         "flavor",
#         "beverage_category",
#         "packaging_type",
#         "packaging_size",
#         "effective_unit_price",
#         "effective_unit_profit",
#         "effective_unit_cost",
#         "abandoned"
#     ]
# ].sort_values(["customer_id", "event_ts_utc", "item_id"])

# display(item_activity_data)
# item_activity_data.to_csv("/Volumes/workspace/default/capstone_data/item_activity.csv", index=False)

# del google_analytics, sales, materials, events, items, items_norm
# del events_flat, price_view, full_events, item_activity_data
# import gc; gc.collect()


| **Column Name**           | **Data Type**   | **Description**                                                           | **Example Value**              |
| ------------------------- | --------------- | ------------------------------------------------------------------------- | ------------------------------ |
| **item_activity_id**      | Integer         | Unique identifier for each item activity record.                          | `523803`                       |
| **session_id**            | Integer         | Identifier for the session during which the activity occurred.            | `27597`                        |
| **customer_id**           | Integer         | Unique identifier for the customer associated with the session.           | `500264365`                    |
| **event_ts_utc**          | Timestamp (UTC) | Date and time of the event in Coordinated Universal Time.                 | `2025-03-31T15:59:18.238Z`     |
| **item_id**               | Integer         | Unique identifier for the specific item involved in the activity.         | `104631`                       |
| **trademark**             | String          | Brand or manufacturer name of the item.                                   | `Fizz Factory`                 |
| **flavor**                | String          | Flavor or variant of the product.                                         | `Tangerine`                    |
| **beverage_category**     | String          | Classification of the beverage (e.g., Nonalcoholic, Soft Drink).          | `OTHER NONALCOHOLIC BEVERAGES` |
| **packaging_type**        | String          | Type of container or packaging used for the product.                      | `CO2 Tank`                     |
| **packaging_size**        | String          | Size or capacity of the packaging.                                        | `20 lb`                        |
| **effective_unit_price**  | Decimal (2)     | Price per unit of the product after discounts or adjustments.             | `29.57`                        |
| **effective_unit_profit** | Decimal (2)     | Profit per unit, calculated as price minus cost.                          | `29.43`                        |
| **effective_unit_cost**   | Decimal (2)     | Cost per unit of the product.                                             | `0.14`                         |
| **abandoned**             | Boolean         | Indicates whether the item was abandoned in the transaction (true/false). | `false`                        |


In [0]:
# COMMAND ----------
# Imports and setup

%pip install prophet
%pip install xgboost
%pip install --upgrade threadpoolctl

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import timedelta

# Time-series
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Prophet (install in cluster if needed: pip install prophet)
from prophet import Prophet

# ML
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

np.random.seed(42)

item_activity_data = pd.read_csv("/Volumes/workspace/default/capstone_data/item_activity.csv")

In [0]:
# COMMAND ----------
# Data prep and feature engineering (from item_activity_data)

# Ensure types
df = item_activity_data.copy()
df["event_ts_utc"] = pd.to_datetime(df["event_ts_utc"], utc=True, errors="coerce")
df["abandoned"] = df["abandoned"].astype(bool)

# Monetary columns
for col in ["effective_unit_price", "effective_unit_profit", "effective_unit_cost"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Per-item realized vs potential revenue/profit
df["realized_revenue"] = np.where(df["abandoned"], 0.0, df["effective_unit_price"])
df["potential_revenue"] = df["effective_unit_price"]
df["abandoned_revenue"] = df["potential_revenue"] - df["realized_revenue"]

df["realized_profit"] = np.where(df["abandoned"], 0.0, df["effective_unit_profit"])
df["potential_profit"] = df["effective_unit_profit"]
df["abandoned_profit"] = df["potential_profit"] - df["realized_profit"]

# Calendar aggregations (daily)
df["event_date"] = df["event_ts_utc"].dt.tz_convert("UTC").dt.floor("D")

daily = (
    df.groupby("event_date", as_index=False)[
        ["realized_revenue", "potential_revenue", "abandoned_revenue",
         "realized_profit", "potential_profit", "abandoned_profit"]
    ].sum()
    .sort_values("event_date")
)

# Abandonment rate by day (share of items abandoned)
daily_counts = df.groupby("event_date")["abandoned"].agg(["mean", "size"]).reset_index()
daily = daily.merge(daily_counts, on="event_date", how="left")
daily = daily.rename(columns={"mean": "abandon_rate", "size": "items_cnt"})

display(daily.head())


In [0]:
# COMMAND ----------
# Quick EDA visuals: revenue and abandonment rate trends

fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Realized vs Potential Revenue with trend
axs[0].plot(daily["event_date"], daily["realized_revenue"], label="Realized Revenue")
axs[0].plot(daily["event_date"], daily["potential_revenue"], label="Potential Revenue")
# Add trend for realized revenue
z_realized = np.polyfit(daily.index, daily["realized_revenue"], 1)
trend_realized = np.polyval(z_realized, daily.index)
axs[0].plot(daily["event_date"], trend_realized, color="tab:blue", linestyle="--", label="Realized Trend")
# Add trend for potential revenue
z_potential = np.polyfit(daily.index, daily["potential_revenue"], 1)
trend_potential = np.polyval(z_potential, daily.index)
axs[0].plot(daily["event_date"], trend_potential, color="tab:orange", linestyle="--", label="Potential Trend")
axs[0].set_title("Daily Revenue: Realized vs Potential")
axs[0].set_ylabel("Revenue")
axs[0].legend()

# Abandonment Rate with trend
axs[1].plot(daily["event_date"], daily["abandon_rate"], color="tab:orange")
z_abandon = np.polyfit(daily.index, daily["abandon_rate"], 1)
trend_abandon = np.polyval(z_abandon, daily.index)
axs[1].plot(daily["event_date"], trend_abandon, color="tab:red", linestyle="--", label="Abandonment Trend")
axs[1].set_title("Daily Abandonment Rate")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("Abandonment Rate")
axs[1].legend()

plt.tight_layout()
plt.show()


| **Model Type**       | **Goal**                            | **Example Algorithms**             | **Output/Use**                 |
| -------------------- | ----------------------------------- | ---------------------------------- | ------------------------------ |
| Time-Series Forecast | Quantify revenue impact over time   | ARIMA, SARIMA, Prophet             | Forecasted revenue lost        |
| Classification       | Predict likelihood of abandonment   | Logistic Regression, Random Forest | Drivers of abandonment         |
| Simulation           | Estimate revenue recovery potential | Counterfactual / Uplift            | Impact under what-if scenarios |
| Clustering           | Identify affected segments          | K-Means                            | Product mix changes            |
| Association Rules    | Explore co-abandonment patterns     | Apriori                            | Product pairing insights       |

In [0]:
# COMMAND ----------
# Model 1: SARIMA on daily realized_revenue, with light hyperparameter search

ts = daily[["event_date", "realized_revenue"]].dropna().copy()
ts = ts.set_index("event_date").asfreq("D").fillna(0.0)

# Train/validation split by time (last 20% as validation)
split_idx = int(len(ts) * 0.8)
y_train = ts.iloc[:split_idx]["realized_revenue"]
y_valid = ts.iloc[split_idx:]["realized_revenue"]

# Grid over small ranges for (p,d,q) and (P,D,Q, s=7 for weekly seasonality)
pdq_grid = [(p, d, q) for p in [0, 1, 2] for d in [0, 1] for q in [0, 1, 2]]
seasonal_grid = [(P, D, Q, 7) for P in [0, 1] for D in [0, 1] for Q in [0, 1]]

best_cfg = None
best_aic = np.inf

for (p, d, q) in pdq_grid:
    for (P, D, Q, s) in seasonal_grid:
        try:
            model = SARIMAX(
                y_train,
                order=(p, d, q),
                seasonal_order=(P, D, Q, s),
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            res = model.fit(disp=0)
            if res.aic < best_aic:
                best_aic = res.aic
                best_cfg = ((p, d, q), (P, D, Q, s))
        except Exception:
            continue

print(f"Best SARIMA cfg: {best_cfg}  AIC={best_aic:.1f}")

# Fit best model on train and forecast validation horizon
order, seasonal_order = best_cfg
final_model = SARIMAX(
    y_train,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=0)

fc = final_model.get_forecast(steps=len(y_valid)).predicted_mean
rmse = mean_squared_error(y_valid, fc, squared=False)
print(f"Validation RMSE: {rmse:,.2f}")

fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# SARIMA Forecast: Realized Revenue
axs[0].plot(y_train.index, y_train.values, label="Train")
axs[0].plot(y_valid.index, y_valid.values, label="Actual")
axs[0].plot(y_valid.index, fc.values, label="Forecast")
axs[0].set_title("SARIMA Forecast: Realized Revenue")
axs[0].set_ylabel("Revenue")
axs[0].legend()

# Daily Lost Revenue (Abandonment)
axs[1].plot(daily["event_date"], daily["abandoned_revenue"], color="tab:orange")
axs[1].set_title("Daily Lost Revenue (Abandonment)")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("Lost Revenue")

plt.tight_layout()
plt.show()


In [0]:
# COMMAND ----------

# In-sample residual diagnostics (one-liner panel like R)

_ = final_model.plot_diagnostics(figsize=(12, 8))

# COMMAND ----------

# # Custom in-sample residual views

# # 1) Residuals over time

resid_in = final_model.resid.dropna()

# plt.figure(figsize=(12, 3))
# plt.plot(resid_in.index, resid_in.values)
# plt.title("In-sample residuals over time")
# plt.xlabel("Date")
# plt.ylabel("Residual")
# plt.tight_layout()
# plt.show()

# # 2) Residuals vs fitted

fitted_in = final_model.fittedvalues.reindex(resid_in.index)

# plt.figure(figsize=(6, 4))
# plt.scatter(fitted_in.values, resid_in.values, s=10)
# plt.title("Residuals vs fitted (in-sample)")
# plt.xlabel("Fitted")
# plt.ylabel("Residual")
# plt.tight_layout()
# plt.show()

# # 3) ACF/PACF of residuals

# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# plt.figure(figsize=(6, 3))
# plot_acf(resid_in, lags=40)
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(6, 3))
# plot_pacf(resid_in, lags=40, method="ywm")
# plt.tight_layout()
# plt.show()

# # 4) Ljung–Box test for remaining autocorrelation

# from statsmodels.stats.diagnostic import acorr_ljungbox

# lb = acorr_ljungbox(resid_in, lags=[12, 24], return_df=True)
# print(lb)

# # COMMAND ----------

# # Out-of-sample (validation) residual diagnostics

# # Validation residuals (actual - forecast)

# resid_val = (y_valid - fc).dropna()

# # 1) Residuals over validation window

# plt.figure(figsize=(12, 3))
# plt.plot(resid_val.index, resid_val.values)
# plt.title("Validation residuals over time (y_valid - forecast)")
# plt.xlabel("Date")
# plt.ylabel("Residual")
# plt.tight_layout()
# plt.show()

# # 2) Histogram + simple normality check

# plt.figure(figsize=(6, 4))
# plt.hist(resid_val.values, bins=30, edgecolor="black")
# plt.title("Validation residuals histogram")
# plt.xlabel("Residual")
# plt.ylabel("Count")
# plt.tight_layout()
# plt.show()

# # 3) ACF of validation residuals

# plt.figure(figsize=(6, 3))
# plot_acf(resid_val, lags=40)
# plt.tight_layout()
# plt.show()

# from statsmodels.stats.diagnostic import acorr_ljungbox
# lb_val = acorr_ljungbox(resid_val, lags=[12, 24], return_df=True)
# print(lb_val)

# COMMAND ----------

# Optional: overlay actual vs fitted on the TRAIN to visually see residual patterns

plt.figure(figsize=(12, 4))
plt.plot(y_train.index, y_train.values, label="Train actual")
plt.plot(fitted_in.index, fitted_in.values, label="Train fitted")
plt.title("Train: actual vs fitted")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

### Notes

#* `plot_diagnostics` is the quickest way to mirror R-style residual panels.
#* Use the Ljung–Box p-values (look at lags like 12, 24) to see if residual
#  autocorrelation remains. Large p-values ⇒ residuals look like white noise.
#* If you see structure in residuals (ACF spikes, patterns vs. fitted), consider
#  adjusting seasonal order, adding exogenous regressors (`exog`), or transforming
#  the series.



In [0]:
# COMMAND ----------
# Model 2: Prophet alternative forecast on realized_revenue (simple tuning)

# Prepare Prophet frame
prophet_df = daily[["event_date", "realized_revenue"]].rename(
    columns={"event_date": "ds", "realized_revenue": "y"}
)
prophet_df = prophet_df.sort_values("ds")
prophet_df["ds"] = pd.to_datetime(prophet_df["ds"]).dt.tz_localize(None)

# Simple time cut (last 20% for validation)
cut_idx = int(len(prophet_df) * 0.8)
train_p = prophet_df.iloc[:cut_idx].copy()
valid_p = prophet_df.iloc[cut_idx:].copy()

# Small grid over changepoint/seasonality settings
param_grid = [
    {"changepoint_prior_scale": 0.05, "seasonality_mode": "additive"},
    {"changepoint_prior_scale": 0.5, "seasonality_mode": "additive"},
    {"changepoint_prior_scale": 0.05, "seasonality_mode": "multiplicative"},
    {"changepoint_prior_scale": 0.5, "seasonality_mode": "multiplicative"},
]

best_rmse = np.inf
best_params = None
best_valid = None

for params in param_grid:
    m = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=params["changepoint_prior_scale"],
        seasonality_mode=params["seasonality_mode"],
    )
    m.fit(train_p)
    future = m.make_future_dataframe(periods=len(valid_p), freq="D")
    fcst = m.predict(future)
    pred = fcst.tail(len(valid_p))["yhat"].values
    rmse_p = mean_squared_error(valid_p["y"].values, pred, squared=False)

    if rmse_p < best_rmse:
        best_rmse = rmse_p
        best_params = params
        best_valid = (valid_p["ds"], valid_p["y"].values, pred)

print(f"Best Prophet params: {best_params}  RMSE={best_rmse:,.2f}")

# Plot best Prophet validation
ds_valid, y_true, y_hat = best_valid
plt.figure(figsize=(10, 4))
plt.plot(train_p["ds"], train_p["y"], label="Train")
plt.plot(ds_valid, y_true, label="Actual")
plt.plot(ds_valid, y_hat, label="Forecast")
plt.title("Prophet Forecast: Realized Revenue")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.show()


In [0]:
# COMMAND ----------
# Refit best Prophet model and get fitted values on TRAIN and forecasts on VALID

from prophet import Prophet
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.graphics.gofplots import qqplot
from statsmodels.stats.diagnostic import acorr_ljungbox

# Refit using best_params discovered above
m_best = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_mode=best_params["seasonality_mode"],
)
m_best.fit(train_p)

# In-sample fitted values
fitted_train = (
    m_best.predict(train_p)[["ds", "yhat"]]
    .rename(columns={"yhat": "yhat_train"})
)
train_with_fit = train_p.merge(fitted_train, on="ds", how="left")

# Out-of-sample (validation) forecast already computed as y_hat
val_df = pd.DataFrame({"ds": ds_valid, "y_true": y_true, "y_hat": y_hat})

# Residuals
resid_in = (train_with_fit["y"] - train_with_fit["yhat_train"]).dropna()
resid_in_index = train_with_fit.loc[resid_in.index, "ds"]

resid_val = (val_df["y_true"] - val_df["y_hat"]).dropna()
resid_val_index = val_df.loc[resid_val.index, "ds"]

print(
    f"In-sample residual mean={resid_in.mean():,.4f}, std={resid_in.std():,.4f}  |  "
    f"Validation residual mean={resid_val.mean():,.4f}, std={resid_val.std():,.4f}"
)

# COMMAND ----------
# Quick visuals like R: residuals over time, vs fitted, histogram, QQ, ACF/PACF

# In-sample residuals over time
plt.figure(figsize=(12, 3))
plt.plot(resid_in_index, resid_in.values)
plt.title("Prophet (train) residuals over time")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

# Validation residuals over time
plt.figure(figsize=(12, 3))
plt.plot(resid_val_index, resid_val.values)
plt.title("Prophet (validation) residuals over time")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

# In-sample residuals vs fitted
plt.figure(figsize=(6, 4))
plt.scatter(
    train_with_fit.loc[resid_in.index, "yhat_train"].values,
    resid_in.values,
    s=10,
)
plt.title("Prophet (train) residuals vs fitted")
plt.xlabel("Fitted (yhat)")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

# Histograms
plt.figure(figsize=(6, 4))
plt.hist(resid_in.values, bins=30, edgecolor="black")
plt.title("Prophet (train) residuals histogram")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(resid_val.values, bins=30, edgecolor="black")
plt.title("Prophet (validation) residuals histogram")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# QQ plots
qqplot(resid_in, line="s")
plt.title("QQ plot — Prophet (train) residuals")
plt.tight_layout()
plt.show()

qqplot(resid_val, line="s")
plt.title("QQ plot — Prophet (validation) residuals")
plt.tight_layout()
plt.show()

# # ACF / PACF
# plt.figure(figsize=(6, 3))
# plot_acf(resid_in, lags=40)
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(6, 3))
# plot_pacf(resid_in, lags=40, method="ywm")
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(6, 3))
# plot_acf(resid_val, lags=40)
# plt.tight_layout()
# plt.show()

# Ljung–Box tests (look for large p-values ⇒ white noise)
print("Ljung–Box (train):")
print(acorr_ljungbox(resid_in, lags=[12, 24], return_df=True))
print("\nLjung–Box (validation):")
print(acorr_ljungbox(resid_val, lags=[12, 24], return_df=True))

# COMMAND ----------
# Prophet component plots (trend/weekly/yearly) for context

# Predict over full window (train+valid) to visualize components
future_all = pd.concat(
    [train_p[["ds"]], val_df[["ds"]]],
    ignore_index=True
).drop_duplicates()
fcst_all = m_best.predict(future_all)

_ = m_best.plot(fcst_all)
plt.title("Prophet: overall fit")
plt.tight_layout()
plt.show()

_ = m_best.plot_components(fcst_all)
plt.tight_layout()
plt.show()

# COMMAND ----------
# Prophet cross-validation utility for rolling-origin residuals
# Fix: make horizon/initial/period "safe" so Prophet doesn't raise:
# ValueError: Less data than horizon after initial window.

from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

# Compute safe initial, horizon, and period in days (assuming daily frequency)
total_days = len(train_p) + len(valid_p)

# Start with conservative splits
initial_days = max(1, min(len(train_p), int(total_days * 0.7)))
horizon_days = max(
    1,
    min(len(valid_p), total_days - initial_days - 1)
)

# Ensure there is room for at least one cutoff
if initial_days + horizon_days >= total_days:
    horizon_days = max(1, total_days - initial_days - 2)

# Ensure period is <= horizon and at least 1
period_days = max(1, min(30, horizon_days // 2 if horizon_days >= 2 else 1))

print(
    f"CV settings → initial={initial_days} days, "
    f"horizon={horizon_days} days, period={period_days} days"
)

# Attempt CV, progressively relax horizon/period if Prophet still complains
cv_df = None
_attempts = 0
while cv_df is None and _attempts < 5:
    try:
        cv_df = cross_validation(
            m_best,
            horizon=f"{horizon_days} days",
            initial=f"{initial_days} days",
            period=f"{period_days} days",
            parallel="processes",
        )
    except ValueError as e:
        _attempts += 1
        # Shrink horizon and period to make a feasible schedule
        if horizon_days > 1:
            horizon_days = max(1, horizon_days - 1)
        period_days = max(1, min(period_days, horizon_days // 2 if horizon_days >= 2 else 1))
        print(
            f"Adjusted CV due to: {e}\n"
            f"Retry {_attempts}: initial={initial_days}, "
            f"horizon={horizon_days}, period={period_days}"
        )

if cv_df is None:
    raise RuntimeError(
        "Prophet CV could not be scheduled with the available data. "
        "Try shortening the horizon, reducing period, or increasing the training window."
    )

perf = performance_metrics(cv_df)
display(perf.head())

# Residuals from CV
cv_resid = cv_df["y"] - cv_df["yhat"]

plt.figure(figsize=(12, 3))
plt.plot(cv_df["ds"], cv_resid)
plt.title("Prophet CV residuals over time")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

_ = plot_cross_validation_metric(cv_df, metric="rmse")
plt.tight_layout()
plt.show()

# ---
# Tips:
# * If you see autocorrelation in residuals, consider additional seasonalities
#   (e.g., holidays, promotions) via `add_seasonality` or `add_regressor`.
# * For multiplicative effects, ensure non-negativity and consider
#   `seasonality_mode="multiplicative"` as you already grid-searched.


In [0]:
# COMMAND ----------
# Model 3: Logistic Regression to predict abandonment (with simple tuning)

# Simple L2-regularized logistic model with sparse-safe preprocessing and
# randomized search. This keeps everything sparse and uses a faster solver.

from pathlib import Path
from joblib import Memory
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from scipy.stats import loguniform

# Compute top-k bucket mapping once per column
def top_k_bucket(s: pd.Series, k: int) -> pd.Series:
    """Bucket infrequent categories into '__other__'."""
    top = s.value_counts(dropna=True).head(k).index
    return s.where(s.isin(top), "__other__")

top_k = 15

# Apply bucketing in place to avoid a full-frame copy
cls = df.copy()
for col in ["trademark", "flavor", "beverage_category",
            "packaging_type", "packaging_size"]:
    if col in cls.columns:
        cls[col] = top_k_bucket(cls[col].astype(str), top_k)

# Define target and features
y = cls["abandoned"].astype(int)
X = cls[["effective_unit_price", "effective_unit_cost",
         "effective_unit_profit", "trademark", "flavor",
         "beverage_category", "packaging_type", "packaging_size"]].copy()

num_cols = ["effective_unit_price", "effective_unit_cost",
            "effective_unit_profit"]
cat_cols = ["trademark", "flavor", "beverage_category",
            "packaging_type", "packaging_size"]

# Sparse-safe preprocessing
pre = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              sparse_output=True), cat_cols),
    ],
    sparse_threshold=1.0
)

# Faster solver on sparse data
logreg = LogisticRegression(
    solver="saga",
    penalty="l2",
    class_weight="balanced",
    max_iter=2000,
    tol=1e-3,
    random_state=42
)

# Cache preprocessing to avoid re-fitting OHE each fold
memory = Memory(location=Path("./.sk_cache"), verbose=0)

pipe = Pipeline(
    steps=[("pre", pre), ("clf", logreg)],
    memory=memory
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Randomized search over C (log-uniform). Add L1 if needed.
param_dist = {
    "clf__C": loguniform(1e-2, 1e2),
    # Optionally search penalty: ["l1", "l2"] with saga; keep l2 for speed first.
    # "clf__penalty": ["l1", "l2"],
}

cv = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    random_state=42
)

cv.fit(X_train, y_train)

best_lr = cv.best_estimator_
y_prob = best_lr.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Logistic Regression ROC")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()

print(f"Best params: {cv.best_params_}")
print(f"Test AUC: {auc:.3f}")


In [0]:
# COMMAND ----------
# Model 4: Gradient Boosted Trees (XGBoost) for abandonment (with tuning)

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist"
)

pipe_xgb = Pipeline([("pre", pre), ("clf", xgb)])

param_grid_xgb = {
    "clf__n_estimators": [150, 300],
    "clf__max_depth": [3, 6],
    "clf__learning_rate": [0.05, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0],
}

cv_xgb = GridSearchCV(
    pipe_xgb, param_grid=param_grid_xgb, cv=3, scoring="roc_auc", n_jobs=-1
)
cv_xgb.fit(X_train, y_train)

print(f"Best XGB params: {cv_xgb.best_params_}")
best_xgb = cv_xgb.best_estimator_

y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
print(f"XGB Test AUC: {auc_xgb:.3f}")

fpr, tpr, _ = roc_curve(y_test, y_prob_xgb)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC={auc_xgb:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("XGBoost ROC")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()


In [0]:
# COMMAND ----------
# Model 5: Scenario simulation — revenue impact using predicted probabilities

# Expected lost revenue per item (using best classifier)
# E[loss] = P(abandon) * price, assuming 1 unit per row
probs_full = best_xgb.predict_proba(X)[:, 1]
cls["p_abandon"] = probs_full
cls["expected_lost_revenue"] = cls["p_abandon"] * cls["effective_unit_price"]
cls["expected_lost_profit"] = cls["p_abandon"] * cls["effective_unit_profit"]

scenario_summary = pd.DataFrame({
    "value": [
        cls["realized_revenue"].sum(),
        cls["potential_revenue"].sum(),
        cls["abandoned_revenue"].sum(),
        cls["expected_lost_revenue"].sum(),
        cls["realized_profit"].sum(),
        cls["potential_profit"].sum(),
        cls["expected_lost_profit"].sum(),
    ]
}, index=[
    "actual_realized_revenue",
    "potential_revenue",
    "actual_abandoned_revenue",
    "expected_lost_revenue",
    "actual_realized_profit",
    "potential_profit",
    "expected_lost_profit"
])

display(scenario_summary)

# Simulate improvement: reduce abandonment probability by 10% relative
cls["p_abandon_improved"] = (cls["p_abandon"] * 0.9).clip(0, 1)
cls["exp_lost_rev_improved"] = (
    cls["p_abandon_improved"] * cls["effective_unit_price"]
)
cls["exp_lost_profit_improved"] = (
    cls["p_abandon_improved"] * cls["effective_unit_profit"]
)

delta_rev = (
    cls["expected_lost_revenue"].sum()
    - cls["exp_lost_rev_improved"].sum()
)
delta_profit = (
    cls["expected_lost_profit"].sum()
    - cls["exp_lost_profit_improved"].sum()
)

print(f"Revenue recovered if abandonment falls 10%: ${delta_rev:,.0f}")
print(f"Profit recovered if abandonment falls 10%: ${delta_profit:,.0f}")

# Daily scenario view for planning
daily_scn = (
    cls.groupby("event_date", as_index=False)[
        ["expected_lost_revenue", "exp_lost_rev_improved"]
    ].sum()
)
plt.figure(figsize=(10, 4))
plt.plot(daily_scn["event_date"], daily_scn["expected_lost_revenue"],
         label="Baseline Expected Lost Rev")
plt.plot(daily_scn["event_date"], daily_scn["exp_lost_rev_improved"],
         label="Improved Scenario")
plt.title("Daily Expected Lost Revenue: Baseline vs Improved")
plt.xlabel("Date")
plt.ylabel("Expected Lost Revenue")
plt.legend()
plt.show()


In [0]:
# COMMAND ----------
# Model 6 (Optional): Product mix shift — cluster items by abandonment & profit

# Item-level rollup
item_mix = (
    df.groupby("item_id").agg(
        n_items=("item_id", "size"),
        abandon_rate=("abandoned", "mean"),
        mean_price=("effective_unit_price", "mean"),
        mean_profit=("effective_unit_profit", "mean"),
        category=("beverage_category", lambda x: x.mode().iloc[0]
                  if len(x.mode()) else "__unknown__"),
    )
    .reset_index()
)

# KMeans clustering on numeric features
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

mix_features = item_mix[["abandon_rate", "mean_price", "mean_profit"]].fillna(0.0)

scaler = StandardScaler()
X_mix = scaler.fit_transform(mix_features)

# Elbow (k from 2 to 6)
inertias = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_mix)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(range(2, 7), inertias, marker="o")
plt.title("Elbow Plot for Item Clusters")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

# Choose k=3 as a starting point
km = KMeans(n_clusters=3, n_init=50, random_state=42)
item_mix["cluster"] = km.fit_predict(X_mix)

# Cluster summary by product category
cluster_summary = (
    item_mix.groupby(["cluster", "category"])
    .agg(n_items=("item_id", "size"),
         avg_abandon_rate=("abandon_rate", "mean"),
         avg_profit=("mean_profit", "mean"))
    .reset_index()
    .sort_values(["cluster", "n_items"], ascending=[True, False])
)

display(cluster_summary.head(30))

# Visualize cluster means
cluster_means = (
    item_mix.groupby("cluster")[["abandon_rate", "mean_price", "mean_profit"]]
    .mean()
    .reset_index()
)

cluster_means.plot(x="cluster", y=["abandon_rate", "mean_price", "mean_profit"],
                   kind="bar", figsize=(8, 4))
plt.title("Cluster Means: Abandon Rate, Price, Profit")
plt.xlabel("Cluster")
plt.ylabel("Value")
plt.tight_layout()
plt.show()


In [0]:
# COMMAND ----------
# Final — concise KPIs for business interpretation

total_realized_rev = daily["realized_revenue"].sum()
total_potential_rev = daily["potential_revenue"].sum()
total_lost_rev = daily["abandoned_revenue"].sum()

total_realized_profit = daily["realized_profit"].sum()
total_potential_profit = daily["potential_profit"].sum()
total_lost_profit = daily["abandoned_profit"].sum()

print(f"Realized Revenue: ${total_realized_rev:,.0f}")
print(f"Potential Revenue (w/o abandonment): ${total_potential_rev:,.0f}")
print(f"Lost Revenue due to abandonment: ${total_lost_rev:,.0f}")
print()
print(f"Realized Profit: ${total_realized_profit:,.0f}")
print(f"Potential Profit (w/o abandonment): ${total_potential_profit:,.0f}")
print(f"Lost Profit due to abandonment: ${total_lost_profit:,.0f}")
